# Medical Framework — Unified Pipeline
Single `imblearn.Pipeline` driven by `RandomizedSearchCV` (200 draws). Each step is one of the custom transformers from the `.py` modules; the search picks the best combination.

In [ ]:
import os
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from imblearn.pipeline import Pipeline
from imblearn import FunctionSampler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_curve, f1_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from loader import load_data

from clean_data import MedianImputer, KNNImputerWrapper, IterativeModelImputer
from tame_outlier import IsolationForestTamer
from normalization import RobustScalerNorm, ZScoreNormalizationNorm
from feature_selection import SelectKBestFilter, TreeBasedSelection
from balance import SMOTESampler, BorderlineSMOTESampler
from model_training import (
    LogisticRegressionEstimator, RandomForestEstimator,
    XGBoostEstimator, LightGBMEstimator, CatBoostEstimator,
)

# imblearn-native no-op sampler — replaces the custom IdentitySampler whose
# `_sampling_type = "bypass"` was the most likely culprit for the NaN-everywhere
# CV scores. FunctionSampler with a pass-through func is officially supported.
def _identity(X, y):
    return X, y

def make_identity_sampler():
    return FunctionSampler(func=_identity, validate=False)

# Clear any stale joblib pipeline cache from previous failed runs. A poisoned
# cache combined with n_jobs>1 was the second suspect for the NaN scores.
shutil.rmtree('./cache', ignore_errors=True)

## Load & split

In [ ]:
path = './Final/data'
typeData = 'csv'
y_column = 'CVD.event'

X, Y, all_mappings, y_mappings = load_data(path=f'{path}.{typeData}', y_column=y_column)
X = X.astype('float32')

print(f'X shape : {X.shape}')
print(f'Classes : {pd.Series(Y).value_counts().to_dict()}')

x_train, x_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y,
)

# Imbalance ratio used to seed scale_pos_weight searches downstream.
neg, pos = (np.array(y_train) == 0).sum(), (np.array(y_train) == 1).sum()
spw_base = float(neg) / float(max(pos, 1))
print(f'neg/pos in train: {neg}/{pos}  ->  scale_pos_weight base ~ {spw_base:.2f}')

## Build the pipeline
Six stages: imputation → outlier flags → normalization → feature selection → balancing → classifier. The starting values are placeholders — `RandomizedSearchCV` swaps each step out below.

In [ ]:
pipeline = Pipeline(steps=[
    ('imputer',    MedianImputer()),
    ('tamer',      'passthrough'),
    ('normalizer', ZScoreNormalizationNorm()),
    ('selector',   SelectKBestFilter(k=20)),
    ('balancer',   make_identity_sampler()),
    ('classifier', LogisticRegressionEstimator()),
])
# NOTE: memory='./cache' removed intentionally. A stale joblib cache with
# n_jobs>1 was a prime suspect for the NaN scores. Re-enable later if needed.
pipeline

## Search space
Each sub-dict pins one classifier and lists compatible step choices + hyper-parameters. `RandomizedSearchCV` samples 200 combinations from the cross-product of all sub-dicts.

In [ ]:
common_imputers    = [MedianImputer(), KNNImputerWrapper(n_neighbors=5), IterativeModelImputer()]
common_tamers      = ['passthrough', IsolationForestTamer()]
common_normalizers = [ZScoreNormalizationNorm(), RobustScalerNorm()]
# When SMOTE is active and the model also uses class_weight='balanced',
# rebalancing is applied twice. We keep both options but flag this for tuning.
common_balancers   = [
    make_identity_sampler(),
    SMOTESampler(k_neighbors=3),
    SMOTESampler(k_neighbors=5),
    BorderlineSMOTESampler(k_neighbors=3),
]

# A calibrated stacker that runs alongside the single-model sub-grids.
# CalibratedClassifierCV ('isotonic') fixes the well-known mis-calibration of
# tree-model probabilities, which directly improves PR-AUC and threshold tuning.
stacking_clf = StackingClassifier(
    estimators=[
        ('xgb', XGBoostEstimator(n_estimators=300, learning_rate=0.05, max_depth=4,
                                 scale_pos_weight=spw_base, subsample=0.9,
                                 colsample_bytree=0.9, reg_lambda=1.0)),
        ('lgbm', LightGBMEstimator(n_estimators=300, learning_rate=0.05,
                                   num_leaves=31, min_child_samples=20,
                                   reg_lambda=1.0)),
        ('lr', LogisticRegressionEstimator(C=1.0, penalty='l2')),
    ],
    final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced',
                                       random_state=42),
    stack_method='predict_proba',
    n_jobs=1,
    passthrough=False,
)
calibrated_stacker = CalibratedClassifierCV(estimator=stacking_clf, method='isotonic', cv=3)

param_grid = [
    # Logistic Regression + SelectKBest
    {
        'imputer':     common_imputers,
        'tamer':       common_tamers,
        'normalizer':  common_normalizers,
        'selector':    [SelectKBestFilter()],
        'selector__k': [10, 20, 30, 40],
        'balancer':    common_balancers,
        'classifier':  [LogisticRegressionEstimator()],
        'classifier__C':       [0.01, 0.1, 1.0, 10.0, 100.0],
        'classifier__penalty': ['l1', 'l2'],
    },
    # Logistic Regression + TreeBased selector
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [LogisticRegressionEstimator()],
        'classifier__C':       [0.01, 0.1, 1.0, 10.0, 100.0],
        'classifier__penalty': ['l1', 'l2'],
    },
    # Random Forest — much wider grid; min_samples_leaf is the strongest unused lever.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [RandomForestEstimator()],
        'classifier__n_estimators':     [200, 400, 600],
        'classifier__max_depth':        [None, 6, 10, 16, 24],
        'classifier__min_samples_leaf': [1, 5, 10, 20],
        'classifier__max_features':     ['sqrt', 'log2', 0.5],
    },
    # XGBoost — now with scale_pos_weight, regularization, subsampling.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [XGBoostEstimator()],
        'classifier__n_estimators':     [200, 400, 600],
        'classifier__learning_rate':    [0.03, 0.05, 0.1],
        'classifier__max_depth':        [3, 5, 7, 9],
        'classifier__min_child_weight': [1, 5, 10],
        'classifier__subsample':        [0.7, 0.85, 1.0],
        'classifier__colsample_bytree': [0.7, 0.85, 1.0],
        'classifier__reg_alpha':        [0.0, 0.1, 1.0],
        'classifier__reg_lambda':       [0.5, 1.0, 5.0],
        'classifier__scale_pos_weight': [1.0, 3.0, spw_base, spw_base * 1.5],
    },
    # LightGBM — wider grid + scale_pos_weight alternative path.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [LightGBMEstimator()],
        'classifier__n_estimators':      [200, 400, 600],
        'classifier__learning_rate':     [0.03, 0.05, 0.1],
        'classifier__num_leaves':        [15, 31, 63, 127],
        'classifier__min_child_samples': [5, 20, 50],
        'classifier__reg_alpha':         [0.0, 0.1, 1.0],
        'classifier__reg_lambda':        [0.0, 0.1, 1.0],
        'classifier__subsample':         [0.7, 0.85, 1.0],
        'classifier__colsample_bytree':  [0.7, 0.85, 1.0],
        'classifier__scale_pos_weight':  [None, 1.0, spw_base],
    },
    # CatBoost — strong on tabular medical data, native imbalance handling.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   [make_identity_sampler(), SMOTESampler(k_neighbors=5)],
        'classifier': [CatBoostEstimator()],
        'classifier__iterations':    [300, 500, 800],
        'classifier__learning_rate': [0.03, 0.05, 0.1],
        'classifier__depth':         [4, 6, 8],
        'classifier__l2_leaf_reg':   [1.0, 3.0, 9.0],
    },
    # Calibrated stacking ensemble — preprocessing is still tunable around it.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   [make_identity_sampler(), SMOTESampler(k_neighbors=5)],
        'classifier': [calibrated_stacker],
    },
]

n_combos = sum(int(np.prod([len(v) for v in g.values()])) for g in param_grid)
print(f'Total candidate configurations (full grid): {n_combos}')

## Fit the search

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Two changes here that matter the most for the diagnosis:
#  * scoring='average_precision' (PR-AUC) — far more informative than ROC-AUC
#    for an 11% prevalence target like CVD.event.
#  * error_score='raise' — was silently np.nan before, which is exactly why every
#    earlier run looked "successful" while in fact every fold was crashing.
#    Once the search runs cleanly you can flip this back to np.nan + re-raise
#    n_jobs to fan out.
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=200,
    scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
    refit='pr_auc',
    cv=cv,
    n_jobs=1,                # bump to 3+ once a clean run is confirmed
    pre_dispatch='n_jobs',
    verbose=2,
    random_state=42,
    return_train_score=False,
    error_score='raise',
)

search.fit(x_train, y_train)

## Inspect the winner

In [ ]:
print(f'Best CV PR-AUC : {search.best_score_:.4f}')
print('Best pipeline   :')
for name, step in search.best_estimator_.named_steps.items():
    label = 'passthrough' if isinstance(step, str) else type(step).__name__
    print(f'  {name:11s} -> {label}')

print('\nBest params:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

# Hold-out scores at the *default* 0.5 threshold (kept for continuity).
y_proba = search.predict_proba(x_val)[:, 1]
y_pred_default = (y_proba >= 0.5).astype(int)

val_roc_auc = roc_auc_score(y_val, y_proba)
val_pr_auc  = average_precision_score(y_val, y_proba)

print(f'\nHold-out ROC-AUC : {val_roc_auc:.4f}')
print(f'Hold-out PR-AUC  : {val_pr_auc:.4f}')
print('\nConfusion matrix @ threshold=0.50 (uninformative on imbalanced data):')
print(confusion_matrix(y_val, y_pred_default))
print(classification_report(y_val, y_pred_default, zero_division=0))

# --------------------------------------------------------------------------
# Post-hoc threshold optimization.
# A model that predicts at 0.5 on 11% prevalence is structurally biased toward
# the majority class. We pick the threshold that maximizes F1 on the holdout's
# precision-recall curve (swap to F2 if recall matters more clinically).
# --------------------------------------------------------------------------
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
# precision_recall_curve returns one extra precision/recall (the trailing 1/0).
f1s = 2 * precisions[:-1] * recalls[:-1] / np.clip(precisions[:-1] + recalls[:-1], 1e-12, None)
best_idx = int(np.nanargmax(f1s))
best_thr = float(thresholds[best_idx])

# F2 alternative (recall-weighted) — print for comparison.
beta = 2.0
f2s = (1 + beta**2) * precisions[:-1] * recalls[:-1] / np.clip(beta**2 * precisions[:-1] + recalls[:-1], 1e-12, None)
best_f2_idx = int(np.nanargmax(f2s))
best_f2_thr = float(thresholds[best_f2_idx])

# Recall@90%-specificity — a clinically common operating point.
fpr, tpr, roc_thr = roc_curve(y_val, y_proba)
spec90_mask = (1 - fpr) >= 0.90
if spec90_mask.any():
    idx90 = int(np.argmax(tpr * spec90_mask))
    recall_at_spec90 = float(tpr[idx90])
    thr_at_spec90    = float(roc_thr[idx90])
else:
    recall_at_spec90, thr_at_spec90 = float('nan'), float('nan')

print('\n--- Operating points ---')
print(f'Best F1 threshold : {best_thr:.4f}   (F1={f1s[best_idx]:.3f}, P={precisions[best_idx]:.3f}, R={recalls[best_idx]:.3f})')
print(f'Best F2 threshold : {best_f2_thr:.4f}  (F2={f2s[best_f2_idx]:.3f}, P={precisions[best_f2_idx]:.3f}, R={recalls[best_f2_idx]:.3f})')
print(f'Recall @ Spec=0.90: {recall_at_spec90:.3f}  (threshold={thr_at_spec90:.4f})')

y_pred_tuned = (y_proba >= best_thr).astype(int)
print(f'\nConfusion matrix @ tuned F1 threshold={best_thr:.4f}:')
print(confusion_matrix(y_val, y_pred_tuned))
print(classification_report(y_val, y_pred_tuned, zero_division=0))

## Leaderboard

In [ ]:
cv_df = (
    pd.DataFrame(search.cv_results_)
      .sort_values('mean_test_pr_auc', ascending=False)
      [['mean_test_pr_auc', 'std_test_pr_auc', 'mean_test_roc_auc', 'std_test_roc_auc', 'params']]
      .head(15)
      .reset_index(drop=True)
)
cv_df

## Persist artifacts

In [ ]:
out_dir = os.path.dirname(path) or '.'
os.makedirs(out_dir, exist_ok=True)

joblib.dump(search.best_estimator_, os.path.join(out_dir, 'pipeline.pkl'))
joblib.dump(all_mappings,           os.path.join(out_dir, 'all_mapping.pkl'))
joblib.dump(y_mappings,             os.path.join(out_dir, 'y_mappings.pkl'))
joblib.dump(list(X.columns),        os.path.join(out_dir, 'features.pkl'))

# Persist the tuned operating point — required for inference, since the default
# 0.5 is the wrong cut for this prevalence.
joblib.dump(
    {
        'threshold_f1':           best_thr,
        'threshold_f2':           best_f2_thr,
        'threshold_spec90':       thr_at_spec90,
        'val_roc_auc':            val_roc_auc,
        'val_pr_auc':             val_pr_auc,
        'recall_at_spec90':       recall_at_spec90,
    },
    os.path.join(out_dir, 'operating_point.pkl'),
)

print('Saved:', os.path.join(out_dir, 'pipeline.pkl'))
print('Saved:', os.path.join(out_dir, 'operating_point.pkl'))